In [18]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

INPUT_DIR = "data/in-hospital-mortality"
OUTPUT_DIR = "data/in-hospital-mortality-cleaned"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


In [19]:
# Load the stays file and build a subject_id -> age mapping
stays = pd.read_csv("data/root/all_stays.csv")
age_map = stays.set_index("SUBJECT_ID")["AGE"].to_dict()

In [20]:
CLEAN_FNS = {
    "Capillary refill rate": "clean_crr",
    "Diastolic blood pressure": "clean_dbp",
    "Systolic blood pressure": "clean_sbp",
    "Fraction inspired oxygen": "clean_fio2",
    "Oxygen saturation": "clean_o2sat",
    "Glucose": "clean_lab",
    "pH": "clean_lab",
    "Temperature": "clean_temperature",
    "Weight": "clean_weight",
    "Height": "clean_height",
    "Glascow coma scale eye opening": "clean_gcs_eye",
    "Glascow coma scale motor response": "clean_gcs_motor",
    "Glascow coma scale verbal response": "clean_gcs_verbal"
}


In [21]:
def clean_sbp(series):
    def extract(v):
        if isinstance(v, str) and "/" in v:
            return float(v.split("/")[0])
        return v
    return pd.to_numeric(series.apply(extract), errors="coerce")


def clean_dbp(series):
    def extract(v):
        if isinstance(v, str) and "/" in v:
            return float(v.split("/")[1])
        return v
    return pd.to_numeric(series.apply(extract), errors="coerce")


def clean_crr(series):
    s = series.astype(str).str.strip()
    mapping = {
        "Normal <3 secs": 0,
        "Brisk": 0,
        "Abnormal >3 secs": 1,
        "Delayed": 1
    }
    mapped = s.map(mapping)
    numeric = pd.to_numeric(series, errors="coerce")
    return mapped.combine_first(numeric)


def clean_fio2(series):
    v = pd.to_numeric(series, errors="coerce")
    idx = v > 1.0
    v.loc[idx] = v.loc[idx] / 100.0
    return v


def clean_lab(series):
    return pd.to_numeric(series, errors="coerce")


def clean_o2sat(series):
    v = pd.to_numeric(series, errors="coerce")
    idx = v <= 1.0
    v.loc[idx] = v.loc[idx] * 100.0
    return v


def clean_temperature(series):
    v = pd.to_numeric(series, errors="coerce")
    idx = v >= 79
    v.loc[idx] = (v.loc[idx] - 32) * 5.0 / 9.0
    return v


def clean_weight(series):
    v = pd.to_numeric(series, errors="coerce")
    idx = v > 250
    v.loc[idx] = v.loc[idx] * 0.453592
    return v


def clean_height(series):
    v = pd.to_numeric(series, errors="coerce")
    idx = (v > 0) & (v < 3)
    v.loc[idx] = v.loc[idx] * 100.0
    return v


def clean_gcs_eye(series):
    s = series.astype(str).str.strip()
    mapping = {
    # 4
    '4 Spontaneously': 4,
    'Spontaneously': 4,

    # 3
    '3 To speech': 3,
    'To Speech': 3,

    # 2
    '2 To pain': 2,
    'To Pain': 2,

    # 1
    '1 No Response': 1
}

    mapped = s.map(mapping)
    numeric = pd.to_numeric(series, errors="coerce")

    # Use mapped where available, fall back to numeric
    result = mapped.copy()
    result[mapped.isna()] = numeric[mapped.isna()]
    return pd.to_numeric(result, errors="coerce")  # ensure float dtype


def clean_gcs_motor(series):
    s = series.astype(str).str.strip()
    mapping = {
    # 6
    '6 Obeys Commands': 6,
    'Obeys Commands': 6,

    # 5
    '5 Localizes Pain': 5,
    'Localizes Pain': 5,

    # 4
    '4 Flex-withdraws': 4,
    'Flex-withdraws': 4,

    # 3
    '3 Abnorm flexion': 3,
    'Abnormal Flexion': 3,

    # 2
    '2 Abnorm extensn': 2,
    'Abnormal extension': 2,

    # 1
    '1 No Response': 1,
    'No response': 1
}
    mapped = s.map(mapping)
    numeric = pd.to_numeric(series, errors="coerce")

    result = mapped.copy()
    result[mapped.isna()] = numeric[mapped.isna()]
    return pd.to_numeric(result, errors="coerce")


def clean_gcs_verbal(series):
    s = series.astype(str).str.strip()
    
    mapping = {
        # Score 5
        'Oriented': 5,
        '5 Oriented': 5,

        # Score 4
        'Confused': 4,
        '4 Confused': 4,

        # Score 3
        'Inappropriate Words': 3,
        '3 Inapprop words': 3,

        # Score 2
        'Incomprehensible sounds': 2,
        '2 Incomp sounds': 2,

        # Score 1
        'No Response': 1,
        '1 No Response': 1,
        'No Response-ETT': 1,
        '1.0 ET/Trach': 1
    }

    mapped = s.map(mapping)
    numeric = pd.to_numeric(series, errors="coerce")

    result = mapped.copy()
    result[mapped.isna()] = numeric[mapped.isna()]
    return pd.to_numeric(result, errors="coerce")

In [22]:
def clean_dataframe(df):
    for col in df.columns:
        if col == "Hours":
            continue
        if col in CLEAN_FNS:
            df[col] = globals()[CLEAN_FNS[col]](df[col])
        else:
            # Warn if the column looks like it has string values (shouldn't happen)
            if df[col].dropna().apply(lambda x: isinstance(x, str)).any():
                print(f"WARNING: Column '{col}' has string values but no cleaner registered. Will become NaN.")
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


In [23]:
def load_variable_ranges(path):
    df = pd.read_csv(path)
    df = df.rename(columns={
        "LEVEL2": "VARIABLE",
        "OUTLIER LOW": "OUTLIER_LOW",
        "VALID LOW": "VALID_LOW",
        "VALID HIGH": "VALID_HIGH",
        "OUTLIER HIGH": "OUTLIER_HIGH"
    })
    df = df.set_index("VARIABLE")
    return df


def clip_variable(series, var_name, ranges):
    if var_name not in ranges.index:
        return series

    r = ranges.loc[var_name]
    v = series.copy()

    v[v < r.OUTLIER_LOW] = np.nan
    v[v > r.OUTLIER_HIGH] = np.nan

    v[v < r.VALID_LOW] = r.VALID_LOW
    v[v > r.VALID_HIGH] = r.VALID_HIGH

    return v


In [24]:
def bin_time_series(df):
    df["time_bin"] = df["Hours"].astype(int)
    df = df.groupby("time_bin").mean(numeric_only=True)

    full_index = pd.Index(range(48), name="time_bin")
    df = df.reindex(full_index)

    df["Hours"] = df.index
    return df.reset_index(drop=True)


In [25]:
def add_time_since_last_obs(series):
    last_seen = -1
    result = []

    for i, val in enumerate(series):
        if not pd.isna(val):
            last_seen = i
            result.append(0)
        else:
            result.append(i - last_seen if last_seen != -1 else np.nan)

    return result


In [26]:
def build_matrix_and_mask(df):
    channels = [c for c in df.columns if c != "Hours"]

    T = len(df)
    F = len(channels)

    data_matrix = np.zeros((T, F))
    mask_matrix = np.zeros((T, F))

    for i, col in enumerate(channels):
        values = df[col].values
        mask = ~pd.isna(values)

        data_matrix[:, i] = np.nan_to_num(values, nan=0.0)
        mask_matrix[:, i] = mask.astype(int)

    return data_matrix, mask_matrix


In [27]:
def preprocess_file(file_path, ranges, age_map):
    df = pd.read_csv(file_path)
    df = clean_dataframe(df)
    df = df[df["Hours"] <= 48].copy()

    for col in df.columns:
        if col != "Hours":
            df[col] = clip_variable(df[col], col, ranges)

    df = df.sort_values("Hours")
    df = bin_time_series(df)

    df_before_fill = df.copy()

    for col in df.columns:
        if col == "Hours":
            continue
        df[col + "_missing"] = df[col].isna().astype(int)
        df[col + "_time_since"] = add_time_since_last_obs(df[col])
        df[col + "_delta"] = df[col].diff()

    data_matrix, mask_matrix = build_matrix_and_mask(df_before_fill)

    for col in df.columns:
        if col != "Hours":
            df[col] = df[col].ffill(limit=3)

    # Extract subject_id from filename and add age
    subject_id = int(file_path.stem.split("_")[0])  # e.g. "123_timeseries.csv" -> 123
    df["age"] = age_map.get(subject_id, np.nan)

    return df, data_matrix, mask_matrix

In [28]:
def process_split(split, ranges, age_map):
    input_dir = os.path.join(INPUT_DIR, split)
    output_dir = os.path.join(OUTPUT_DIR, split)
    matrix_dir = os.path.join(OUTPUT_DIR, f"{split}_matrices")

    Path(output_dir).mkdir(parents=True, exist_ok=True)
    Path(matrix_dir).mkdir(parents=True, exist_ok=True)

    files = [f for f in os.listdir(input_dir) if f.endswith("_timeseries.csv")]

    for f in tqdm(files):
        input_path = Path(os.path.join(input_dir, f))  # use Path so .stem works
        output_path = os.path.join(output_dir, f)

        df, data_matrix, mask_matrix = preprocess_file(input_path, ranges, age_map)

        df.to_csv(output_path, index=False)
        np.save(os.path.join(matrix_dir, f.replace(".csv", "_data.npy")), data_matrix)
        np.save(os.path.join(matrix_dir, f.replace(".csv", "_mask.npy")), mask_matrix)

In [29]:
ranges_path = os.path.join(
    "mimic3benchmark",
    "resources",
    "variable_ranges.csv"
)

ranges = load_variable_ranges(ranges_path)

for split in ["train", "test"]:
    print(f"Processing {split}...")
    process_split(split, ranges, age_map)

print("Preprocessing complete.")


Processing train...


100%|██████████| 17903/17903 [06:41<00:00, 44.54it/s]


Processing test...


100%|██████████| 3236/3236 [01:11<00:00, 45.07it/s]

Preprocessing complete.
